# 🖼️ 02: 멀티모달 - 현실 세계의 입력을 다루다

---

| 항목 | 내용 |
|------|------|
| **목표** | AI가 텍스트 너머의 다양한 입력(이미지, 문서)을 처리하는 흐름을 이해한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 5분 / 전체 시연 12분 |
| **API 키** | ✅ 권장 (없으면 사전 준비된 OCR 결과로 대체) |
| **이전 노트북과의 연결** | LLM이 인터페이스라는 것을 봤다. 이제 그 입력이 텍스트 외에도 다양하다는 것을 본다. |

---

## 🎯 핵심 메시지

> **멀티모달은 '이미지도 읽는다'는 기능이 아닙니다.**  
> **현실 세계의 데이터를 AI 시스템이 처리할 수 있게 됐다는 의미입니다.**

```
영수증 → 경비 처리 자동화
화이트보드 사진 → 회의록 자동 생성  
UI 스크린샷 → 사용자 행동 분석
계약서 스캔본 → 핵심 조항 추출
```

이전에는 사람이 수동으로 해야 했던 작업들입니다.

In [ ]:
!pip install -q openai Pillow requests
print("✅ 설치 완료")

## 1️⃣ 환경 설정

In [ ]:
import os, json, base64
from IPython.display import display, HTML, Image
from PIL import Image as PILImage, ImageDraw, ImageFont
import io

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(api_key=""):
    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if key and key not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=key)
            print("✅ API 모드 (gpt-4o-mini vision 사용)")
            return "api", c
        except:
            pass
    print("💡 로컬 모드 - 사전 준비된 OCR 결과와 Mock 응답 사용")
    print("   (실제 이미지 없이도 파이프라인 흐름을 볼 수 있습니다)")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

## 2️⃣ 샘플 이미지 생성

Pillow로 강의용 샘플 이미지를 직접 생성합니다.  
외부 파일 없이도 멀티모달 파이프라인을 시연할 수 있습니다.

In [ ]:
import os
from PIL import Image as PILImage, ImageDraw

os.makedirs("/content/demo_data", exist_ok=True)

def create_receipt_image() -> str:
    """영수증 이미지를 생성합니다."""
    img = PILImage.new("RGB", (400, 600), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    lines = [
        ("TECHCORE 구내식당", 30, (0,0,0), 20),
        ("2024-11-20  12:34", 60, (80,80,80), 14),
        ("-" * 45, 85, (180,180,180), 12),
        ("김치찌개 세트          8,500", 105, (0,0,0), 13),
        ("제육볶음 세트          9,000", 128, (0,0,0), 13),
        ("아메리카노             3,000", 151, (0,0,0), 13),
        ("물                       무료", 174, (0,0,0), 13),
        ("-" * 45, 200, (180,180,180), 12),
        ("소계              20,500원", 220, (0,0,0), 14),
        ("부가세(10%)        2,050원", 245, (0,0,0), 14),
        ("합계              22,550원", 275, (0,0,128), 16),
        ("-" * 45, 305, (180,180,180), 12),
        ("결제수단: 법인카드", 325, (80,80,80), 13),
        ("카드번호: ****-****-****-1234", 348, (80,80,80), 12),
        ("승인번호: 20241120-8834521", 370, (80,80,80), 12),
        ("감사합니다!", 430, (0,100,0), 16),
    ]

    for text, y, color, size in lines:
        draw.text((20, y), text, fill=color)

    path = "/content/demo_data/receipt.png"
    img.save(path)
    return path

def create_whiteboard_image() -> str:
    """화이트보드 이미지를 생성합니다."""
    img = PILImage.new("RGB", (500, 500), color=(245, 245, 240))
    draw = ImageDraw.Draw(img)

    # 마커로 쓴 느낌
    lines = [
        ("Q4 주요 목표 (2024)", 20, (30,30,200), 18),
        ("="*40, 50, (100,100,100), 12),
        ("[제품] CloudSync v2.3 출시  ✓", 70, (0,130,0), 14),
        ("[제품] DataPulse v1.1 출시  ✓", 100, (0,130,0), 14),
        ("[보안] API 키 정책 v3.1     ✓", 130, (0,130,0), 14),
        ("[채용] 시니어 백엔드 3명    2명완료", 160, (180,100,0), 14),
        ("[채용] ML 엔지니어 2명      진행중", 190, (180,100,0), 14),
        ("", 220, (0,0,0), 12),
        ("핵심 메트릭:", 235, (50,50,50), 15),
        ("- MAU 목표: 50K  → 현재 43K", 260, (200,0,0), 13),
        ("- API 응답속도: < 200ms  ✓", 285, (0,130,0), 13),
        ("- 서비스 가용성: 99.9%   ✓", 310, (0,130,0), 13),
        ("", 335, (0,0,0), 12),
        ("TODO:", 350, (50,50,50), 15),
        ("→ ML엔지니어 채용 마무리 (강하늘)", 375, (0,0,180), 13),
        ("→ MAU 갭 분석 필요 (장유진)", 400, (0,0,180), 13),
    ]

    for text, y, color, size in lines:
        if text:
            draw.text((15, y), text, fill=color)

    path = "/content/demo_data/whiteboard.png"
    img.save(path)
    return path

# 이미지 생성
receipt_path = create_receipt_image()
whiteboard_path = create_whiteboard_image()

print("✅ 샘플 이미지 생성 완료")
print(f"  영수증: {receipt_path}")
print(f"  화이트보드: {whiteboard_path}")

In [ ]:
# 생성된 이미지 미리보기
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, path, title in [
    (axes[0], receipt_path, "영수증 이미지"),
    (axes[1], whiteboard_path, "화이트보드 이미지")
]:
    img = PILImage.open(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.suptitle("멀티모달 데모 - 샘플 이미지", fontsize=16)
plt.tight_layout()
plt.show()

## 3️⃣ 사전 준비된 OCR 결과 (로컬 모드 fallback)

API 모드가 없을 때, 또는 이미지 인식이 어려울 때  
사전 준비된 OCR/설명 텍스트로 파이프라인 흐름을 시연합니다.

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 사전 준비된 이미지 설명 텍스트 (Plan B)
# API 모드에서는 실제 이미지를 LLM에 보내지만
# 로컬/오프라인 모드에서는 이 텍스트를 기반으로 동일한 파이프라인 실행
from helpers.sample_data import MOCK_RECEIPT_OCR, MOCK_WHITEBOARD_OCR
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print("✅ Mock OCR 텍스트 준비 완료")

## 4️⃣ 데모 A: 영수증 → 경비 처리 데이터 추출

영수증 이미지에서 경비 처리에 필요한 정보를 자동 추출합니다.  
이전에는 사람이 수동으로 입력하던 작업입니다.

In [ ]:
def image_to_base64(image_path: str) -> str:
    """이미지를 base64로 인코딩합니다."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def extract_receipt_info_api(image_path: str) -> dict:
    """GPT Vision API로 영수증 정보를 추출합니다."""
    img_b64 = image_to_base64(image_path)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64}"}
                    },
                    {
                        "type": "text",
                        "text": """이 영수증 이미지에서 정보를 추출하여 JSON으로 반환하세요.
형식: {"date": "날짜", "merchant": "상호명", "items": [{"name": "품목", "price": 숫자}], "total": 숫자, "payment_method": "결제수단", "category": "식비|교통비|숙박비|기타"}"""
                    }
                ]
            }
        ],
        response_format={"type": "json_object"},
        max_tokens=500
    )
    return json.loads(response.choices[0].message.content)

def extract_receipt_info_local(ocr_text: str) -> dict:
    """OCR 텍스트에서 영수증 정보를 파싱합니다 (로컬 모드)."""
    # 사전 준비된 결과 반환
    return {
        "date": "2024-11-20",
        "merchant": "TECHCORE 구내식당",
        "items": [
            {"name": "김치찌개 세트", "price": 8500},
            {"name": "제육볶음 세트", "price": 9000},
            {"name": "아메리카노", "price": 3000}
        ],
        "total": 22550,
        "payment_method": "법인카드 ****1234",
        "category": "식비",
        "vat": 2050
    }

# 실행
print("=" * 55)
print("  데모 A: 영수증 → 경비 처리 데이터")
print("=" * 55)

if MODE == "api" and client:
    print("🌐 API 모드: 실제 이미지를 GPT-4o-mini Vision으로 분석")
    try:
        receipt_data = extract_receipt_info_api(receipt_path)
        print("✅ 이미지 분석 성공")
    except Exception as e:
        print(f"⚠️ API 실패: {e} → 로컬 모드로 전환")
        receipt_data = extract_receipt_info_local(MOCK_RECEIPT_OCR)
else:
    print("💡 로컬 모드: OCR 텍스트 기반 파싱")
    receipt_data = extract_receipt_info_local(MOCK_RECEIPT_OCR)

# 결과 출력
import pandas as pd
print("\n📄 추출 결과:")
print(f"  날짜: {receipt_data.get('date')}")
print(f"  상호: {receipt_data.get('merchant')}")
print(f"  카테고리: {receipt_data.get('category')}")
print(f"  합계: {receipt_data.get('total'):,}원")
print(f"  결제: {receipt_data.get('payment_method')}")
print("\n  품목 상세:")
items_df = pd.DataFrame(receipt_data.get('items', []))
if not items_df.empty:
    items_df.columns = ["품목", "금액(원)"]
    display(items_df)

print("\n🎯 이 데이터는 경비 처리 시스템 API에 바로 전송할 수 있습니다.")

## 5️⃣ 데모 B: 화이트보드 사진 → 회의록 + 액션 아이템

In [ ]:
def analyze_whiteboard_api(image_path: str) -> dict:
    """화이트보드 이미지를 분석합니다."""
    img_b64 = image_to_base64(image_path)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64}"}
                    },
                    {
                        "type": "text",
                        "text": """이 화이트보드 내용을 분석하여 JSON으로 반환하세요.
형식: {
  "title": "제목",
  "completed_items": ["완료된 항목"],
  "in_progress_items": ["진행중 항목"],
  "key_metrics": [{"metric": "지표명", "value": "값", "status": "ok|warning|danger"}],
  "action_items": [{"task": "할 일", "owner": "담당자"}]
}"""
                    }
                ]
            }
        ],
        response_format={"type": "json_object"},
        max_tokens=600
    )
    return json.loads(response.choices[0].message.content)

def analyze_whiteboard_local() -> dict:
    """사전 준비된 결과 반환 (로컬 모드)."""
    return {
        "title": "Q4 주요 목표 (2024)",
        "completed_items": [
            "CloudSync v2.3 출시",
            "DataPulse v1.1 출시",
            "API 키 보안 정책 v3.1 시행",
            "시니어 백엔드 엔지니어 2명 채용 완료"
        ],
        "in_progress_items": [
            "ML 엔지니어 채용 (목표 2명 중 진행중)"
        ],
        "key_metrics": [
            {"metric": "MAU", "value": "43K (목표 50K)", "status": "warning"},
            {"metric": "API 응답속도", "value": "< 200ms", "status": "ok"},
            {"metric": "서비스 가용성", "value": "99.9%", "status": "ok"}
        ],
        "action_items": [
            {"task": "ML 엔지니어 채용 마무리", "owner": "강하늘"},
            {"task": "MAU 갭 원인 분석", "owner": "장유진"}
        ]
    }

# 실행
print("=" * 55)
print("  데모 B: 화이트보드 → 구조화된 회의 요약")
print("=" * 55)

if MODE == "api" and client:
    try:
        wb_data = analyze_whiteboard_api(whiteboard_path)
    except Exception as e:
        print(f"⚠️ {e} → 로컬 모드")
        wb_data = analyze_whiteboard_local()
else:
    wb_data = analyze_whiteboard_local()

# HTML로 이쁘게 출력
completed = "".join(f"<li>✅ {i}</li>" for i in wb_data.get("completed_items", []))
in_progress = "".join(f"<li>🔄 {i}</li>" for i in wb_data.get("in_progress_items", []))

metrics_html = ""
for m in wb_data.get("key_metrics", []):
    color = {"ok": "#27ae60", "warning": "#e67e22", "danger": "#e74c3c"}.get(m["status"], "#333")
    metrics_html += f'<li><span style="color:{color};font-weight:bold;">{m["metric"]}</span>: {m["value"]}</li>'

actions = "".join(f"<li>📌 {a['task']} <em>→ {a['owner']}</em></li>" for a in wb_data.get("action_items", []))

display(HTML(f"""
<div style="font-family:Arial,sans-serif;max-width:700px;border:1px solid #ddd;border-radius:8px;overflow:hidden;">
  <div style="background:#2c3e50;color:white;padding:12px 16px;">
    <strong>🖼️ {wb_data.get('title', '')}</strong>
    <span style="font-size:12px;float:right;">화이트보드 → 자동 구조화</span>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:0;">
    <div style="padding:15px;border-right:1px solid #eee;">
      <strong>완료 항목</strong>
      <ul style="font-size:13px;line-height:1.8;padding-left:20px;">{completed}</ul>
      <strong>진행 중</strong>
      <ul style="font-size:13px;line-height:1.8;padding-left:20px;">{in_progress}</ul>
    </div>
    <div style="padding:15px;">
      <strong>핵심 지표</strong>
      <ul style="font-size:13px;line-height:1.8;padding-left:20px;">{metrics_html}</ul>
      <strong>액션 아이템</strong>
      <ul style="font-size:13px;line-height:1.8;padding-left:20px;">{actions}</ul>
    </div>
  </div>
</div>
"""))

## 6️⃣ 데모 C: UI 스크린샷 → 사용자 행동 분석

UX 리서치나 제품 개선에 활용할 수 있는 패턴입니다.

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# UI 스크린샷 분석 (실제 이미지 대신 설명 텍스트 활용)
from helpers.sample_data import UI_DESCRIPTION, mock_ux
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

UX_ANALYSIS_PROMPT = f"""아래 UI 스크린샷 설명을 바탕으로 사용자 행동을 분석하세요.

반환 형식 JSON:
{{
  "user_intent": "사용자가 원하는 것 (1문장)",
  "friction_points": ["문제가 되는 UX 요소"],
  "ignored_features": ["무시되는 기능"],
  "recommendations": ["개선 제안"]
}}

스크린샷:
{UI_DESCRIPTION}"""

if MODE == "api" and client:
    try:
        ux_result = json.loads(
            client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"user","content":UX_ANALYSIS_PROMPT}],
                response_format={"type":"json_object"},
                max_tokens=500
            ).choices[0].message.content
        )
    except Exception as e:
        print(f"⚠️ {e}")
        ux_result = mock_ux
else:
    ux_result = mock_ux

print("=" * 55)
print("  데모 C: UI 스크린샷 → 사용자 행동 분석")
print("=" * 55)
print(f"\n🎯 사용자 의도: {ux_result.get('user_intent')}")
print(f"\n⚠️  마찰 요인:")
for f in ux_result.get('friction_points', []):
    print(f"   • {f}")
print(f"\n🚫 무시되는 기능:")
for i in ux_result.get('ignored_features', []):
    print(f"   • {i}")
print(f"\n💡 개선 제안:")
for r in ux_result.get('recommendations', []):
    print(f"   → {r}")

## 7️⃣ 멀티모달 파이프라인 전체 그림

In [ ]:
display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:800px;margin:10px auto;">
  <h3 style="color:#2c3e50;">멀티모달 파이프라인: 입력 유형별 처리</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">입력 유형</th>
        <th style="padding:10px;">기존 방식</th>
        <th style="padding:10px;">멀티모달 AI</th>
        <th style="padding:10px;">출력</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">🧾 영수증 사진</td>
        <td style="padding:10px;">수동 입력</td>
        <td style="padding:10px;">Vision + 구조화</td>
        <td style="padding:10px;">경비처리 JSON</td>
      </tr>
      <tr>
        <td style="padding:10px;">📋 화이트보드</td>
        <td style="padding:10px;">수동 타이핑</td>
        <td style="padding:10px;">OCR + 요약</td>
        <td style="padding:10px;">구조화된 회의록</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">📱 UI 스크린샷</td>
        <td style="padding:10px;">UX 리서처 분석</td>
        <td style="padding:10px;">행동 패턴 분석</td>
        <td style="padding:10px;">개선 제안 리포트</td>
      </tr>
      <tr>
        <td style="padding:10px;">📄 PDF 계약서</td>
        <td style="padding:10px;">변호사 검토</td>
        <td style="padding:10px;">조항 추출 + 위험도</td>
        <td style="padding:10px;">핵심 조항 요약</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">📊 차트 이미지</td>
        <td style="padding:10px;">수동 데이터 입력</td>
        <td style="padding:10px;">데이터 추출 + 해석</td>
        <td style="padding:10px;">수치 데이터 + 인사이트</td>
      </tr>
    </tbody>
  </table>
  <div style="margin-top:15px;padding:12px;background:#fff9c4;border-left:4px solid #f39c12;font-size:13px;">
    💡 공통점: 모두 <strong>'비정형 입력 → 구조화된 데이터'</strong> 변환입니다.<br>
    이 구조화된 데이터가 다음 시스템(DB, API, 자동화 워크플로우)으로 이어집니다.
  </div>
</div>
"""))

---

## 8️⃣ 한계와 주의점

In [ ]:
print("⚠️  멀티모달의 현실적 한계")
print("─" * 50)
print()
print("1. 이미지 품질에 민감")
print("   흐릿한 사진, 반사광, 각도 → 인식률 급락")
print()
print("2. 한국어 OCR 정확도")
print("   영문에 비해 한국어 손글씨 인식은 여전히 도전적")
print()
print("3. API 비용")
print("   이미지 처리는 텍스트 처리보다 토큰 비용 높음")
print("   고해상도 이미지 → 많은 토큰 소비")
print()
print("4. 민감 정보")
print("   영수증, 계약서 등에는 개인정보가 포함될 수 있음")
print("   외부 API 전송 전 마스킹 필요")
print()
print("5. 환각 (Hallucination)")
print("   이미지에 없는 숫자를 만들어낼 수 있음")
print("   금융/법무 문서는 반드시 사람 검증 필요")

---

## 🎤 강의자 멘트 포인트

> **"멀티모달의 핵심은 '이미지도 읽는다'는 기능이 아닙니다.**  
> **현실 세계에 존재하는 모든 데이터를 AI 시스템의 입력으로 만들 수 있다는 의미입니다.**  
>
> 영수증은 더 이상 손으로 입력하지 않아도 됩니다.  
> 화이트보드는 더 이상 사진만 찍어두는 게 아니라 즉시 회의록이 됩니다.  
> 이게 AI가 업무 흐름 안으로 들어오는 방식입니다."

## 🙋 청중 질문 유도
> - "여러분 업무에서 매일 보는 비정형 데이터는 무엇인가요? (영수증, 보고서, 채팅 스크린샷...)"
> - "이 파이프라인을 실제 서비스에 넣는다면 어떤 리스크를 고려해야 할까요?"
> - "로컬 경량 모델 vs 클라우드 API - 어떤 상황에서 어떤 걸 쓸까요?"

## 🏗️ 실무 확장 포인트
- **보안**: 이미지 내 개인정보 마스킹 (PII 검출) 파이프라인
- **정확도**: 2차 검증 레이어 (사람 in the loop) 추가
- **비용 최적화**: 이미지 리사이징 → 토큰 절감
- **온프레미스**: 데이터 외부 전송 불가 환경에서는 Llama 3.2 Vision 같은 로컬 모델 활용

## ➡️ 다음 노트북
**03_embeddings_and_vector_search.ipynb** - 모델이 내 문서를 모른다는 문제, 검색으로 해결하기